In [23]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [24]:
#from google.colab import drive
#drive.mount('/content/drive')

# **Loading Dataset**

In [25]:
train = pd.read_csv("./training_data.csv")
test = pd.read_csv("./testing_data.csv")

#train = pd.read_csv("/content/training_data.csv")
#test = pd.read_csv("/content/testing_data.csv")


print(train.shape)
print(test.shape)
print(train.head())

(32951, 21)
(8237, 21)
   age        job  marital    education  default housing loan    contact  \
0   56  housemaid  married     basic.4y       no      no   no  telephone   
1   57   services  married  high.school  unknown      no   no  telephone   
2   37   services  married  high.school       no     yes   no  telephone   
3   40     admin.  married     basic.6y       no      no   no  telephone   
4   56   services  married  high.school       no      no  yes  telephone   

  month day_of_week  ...  campaign  pdays  previous     poutcome emp.var.rate  \
0   may         mon  ...         1    999         0  nonexistent          1.1   
1   may         mon  ...         1    999         0  nonexistent          1.1   
2   may         mon  ...         1    999         0  nonexistent          1.1   
3   may         mon  ...         1    999         0  nonexistent          1.1   
4   may         mon  ...         1    999         0  nonexistent          1.1   

   cons.price.idx  cons.conf.idx 

# **Check for Missing/Unkown Values**

In [26]:
print(train.dtypes)

print("\n=== MISSING / UNKNOWN COUNTS ===")
for col in train.columns:
    unknowns = (train[col] == 'unknown').sum()
    nulls = train[col].isnull().sum()
    if unknowns > 0 or nulls > 0:
        print(f"  {col}: {nulls} nulls, {unknowns} unknowns")

age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object

=== MISSING / UNKNOWN COUNTS ===
  job: 0 nulls, 269 unknowns
  marital: 0 nulls, 63 unknowns
  education: 0 nulls, 1398 unknowns
  default: 0 nulls, 6916 unknowns
  housing: 0 nulls, 804 unknowns
  loan: 0 nulls, 804 unknowns


In [27]:
# Check if housing and loan has overlapping "Unknown" values
overlap = ((train['housing'] == 'unknown') & (train['loan'] == 'unknown')).sum()
print(f"Numbers of Rows where BOTH housing and loan are unknown: {overlap}")
print(f"Housing unknowns: {(train['housing'] == 'unknown').sum()}")
print(f"Loan unknowns: {(train['loan'] == 'unknown').sum()}")


print("=== TARGET DISTRIBUTION ===") #y is marketing success (Yes/No)
print(train['y'].value_counts())
print(train['y'].value_counts(normalize=True).round(3))


print("=== NUMERIC SUMMARY ===")
num_cols = train.select_dtypes(include='number').columns.tolist()
print(train[num_cols].describe().round(2))

Numbers of Rows where BOTH housing and loan are unknown: 804
Housing unknowns: 804
Loan unknowns: 804
=== TARGET DISTRIBUTION ===
y
no     29239
yes     3712
Name: count, dtype: int64
y
no     0.887
yes    0.113
Name: proportion, dtype: float64
=== NUMERIC SUMMARY ===
            age  duration  campaign     pdays  previous  emp.var.rate  \
count  32951.00  32951.00  32951.00  32951.00  32951.00      32951.00   
mean      40.04    257.17      2.55    961.84      0.17          0.08   
std       10.43    258.69      2.76    188.46      0.49          1.57   
min       17.00      0.00      1.00      0.00      0.00         -3.40   
25%       32.00    102.00      1.00    999.00      0.00         -1.80   
50%       38.00    179.00      2.00    999.00      0.00          1.10   
75%       47.00    318.00      3.00    999.00      0.00          1.40   
max       98.00   4918.00     56.00    999.00      6.00          1.40   

       cons.price.idx  cons.conf.idx  euribor3m  nr.employed  
count     

In [28]:
econ_cols = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
print(train[econ_cols].corr().round(2))

                emp.var.rate  cons.price.idx  cons.conf.idx  euribor3m  \
emp.var.rate            1.00            0.77           0.20       0.97   
cons.price.idx          0.77            1.00           0.06       0.69   
cons.conf.idx           0.20            0.06           1.00       0.28   
euribor3m               0.97            0.69           0.28       1.00   
nr.employed             0.91            0.52           0.10       0.95   

                nr.employed  
emp.var.rate           0.91  
cons.price.idx         0.52  
cons.conf.idx          0.10  
euribor3m              0.95  
nr.employed            1.00  


# **Data Cleaning/Preprocessing**

In [29]:
train_df = train.copy()
test_df = test.copy()

# Drop 'duration' column — target leakage because duration is known after the call ends and by that time the outcome of y is already captured.
#Including "duration" column will inflate the prediction accuracy of y since longer duration will most likely mean a positive outcome.
#Since the goal here is to build a robust model, removing 'duration' is an important step.
train_df.drop(columns=['duration'], inplace=True)
test_df.drop(columns=['duration'], inplace=True)

# drop highly correlated economic indicators
# emp.var.rate (r=0.97 with euribor3m) and nr.employed (r=0.95 with euribor3m) and can complicate model learning
# We retain euribor3m as it is the most granular (daily) economic indicator.
#train_df.drop(columns=['emp.var.rate', 'nr.employed'], inplace=True)
#test_df.drop(columns=['emp.var.rate', 'nr.employed'], inplace=True)

# Engineer 'was_contacted_before' binary flag. pdays=999 means the client was NEVER contacted in a previous campaign.
# This binary flag explicitly captures that distinction, which is highly
# predictive — previously contacted clients behave very differently.
train_df['was_contacted_before'] = (train_df['pdays'] != 999).astype(int)
test_df['was_contacted_before']  = (test_df['pdays'] != 999).astype(int)

# Cap 'campaign' at 95th percentile. Campaign has extreme outliers (max=56, mean=2.5).
#Outliers can distort distance-based models. Capping at the 95th percentile reduces their influence while preserving the distribution shape.
cap_value = train_df['campaign'].quantile(0.95)
train_df['campaign'] = train_df['campaign'].clip(upper=cap_value)
test_df['campaign']  = test_df['campaign'].clip(upper=cap_value)
print(f"Campaign capped at: {cap_value}")

# Encode target variable. Convert y from string ("yes"/"no") to binary integer (1/0).
train_df['y'] = (train_df['y'] == 'yes').astype(int)
test_df['y']  = (test_df['y'] == 'yes').astype(int)

#Train-test split
X_train = train_df.drop(columns=['y'])
y_train = train_df['y']
X_test  = test_df.drop(columns=['y'])
y_test  = test_df['y']

# One-hot encode categorical variables. We one-hot encode all categoricals including "unknown" as its own category since "unknown" may itself carry predictive signal
# drop_first=False keeps all categories for interpretability.
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns to encode: {cat_cols}")

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=False)
X_test  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=False)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
print(f"Features after encoding — Train: {X_train.shape}, Test: {X_test.shape}")

# Scaling and SMOTE done BEFORE cross-validation causes data leakage because information from the full training set leaks into each validation fold.
# Instead, scaling and SMOTE must happen INSIDE the cross val pipeline so they are fit only on each fold's training split.
X_train_ready = X_train.copy()
X_test_ready  = X_test.copy()

print(f"\nClass distribution before SMOTE: {y_train.value_counts().to_dict()}")
print(f"Final shapes — X_train: {X_train_ready.shape}, X_test: {X_test_ready.shape}")

Campaign capped at: 7.0
Categorical columns to encode: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Features after encoding — Train: (32951, 63), Test: (8237, 63)

Class distribution before SMOTE: {0: 29239, 1: 3712}
Final shapes — X_train: (32951, 63), X_test: (8237, 63)


In [30]:
X_train.head

<bound method NDFrame.head of        age  campaign  pdays  previous  emp.var.rate  cons.price.idx  \
0       56         1    999         0           1.1          93.994   
1       57         1    999         0           1.1          93.994   
2       37         1    999         0           1.1          93.994   
3       40         1    999         0           1.1          93.994   
4       56         1    999         0           1.1          93.994   
...    ...       ...    ...       ...           ...             ...   
32946   73         1    999         0          -1.1          94.767   
32947   46         1    999         0          -1.1          94.767   
32948   56         2    999         0          -1.1          94.767   
32949   44         1    999         0          -1.1          94.767   
32950   74         3    999         1          -1.1          94.767   

       cons.conf.idx  euribor3m  nr.employed  was_contacted_before  ...  \
0              -36.4      4.857       5191

# **Section 1: Maximize Test Accuracy (No Class Balancing)**

All models are trained **without any class-balancing techniques** (no SMOTE, no `class_weight`, no `scale_pos_weight`, no `is_unbalance`). The sole optimization target is **accuracy**. This deliberately exposes the trade-off: pursuing raw accuracy on a heavily imbalanced dataset causes sensitivity (true positive rate for "yes") to collapse, as the model learns to over-predict the majority class.

In [31]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score, recall_score, roc_auc_score, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb
import subprocess

# Scoring dict — reported for all models; accuracy drives Section 1 selection
scoring = {
    'accuracy':          make_scorer(accuracy_score),
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'sensitivity':       make_scorer(recall_score, pos_label=1),
    'specificity':       make_scorer(recall_score, pos_label=0),
    'auc':               make_scorer(roc_auc_score),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def gpu_available():
    try:
        subprocess.check_output(['nvidia-smi'], stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

USE_GPU = gpu_available()
print("GPU available:", USE_GPU)
xgb_device = 'cuda' if USE_GPU else 'cpu'
lgb_device  = 'gpu'  if USE_GPU else 'cpu'

neg, pos = np.bincount(y_train)
scale_pos = neg / pos  # stored for Section 2; NOT used here

# Section 1: No class-balancing in any pipeline
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    'Random Forest': Pipeline([
        ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ]),
    'XGBoost': Pipeline([
        ('model', xgb.XGBClassifier(
            eval_metric='logloss', random_state=42, n_jobs=-1, device=xgb_device
        )),
    ]),
    'LightGBM': Pipeline([
        ('model', lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, device=lgb_device)),
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
    ]),
}

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    cv_res = cross_validate(model, X_train_ready, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results[name] = {
        'Accuracy':      cv_res['test_accuracy'].mean().round(4),
        'Balanced Acc':  cv_res['test_balanced_accuracy'].mean().round(4),
        'Sensitivity':   cv_res['test_sensitivity'].mean().round(4),
        'Specificity':   cv_res['test_specificity'].mean().round(4),
        'AUC':           cv_res['test_auc'].mean().round(4),
    }
    print("  ✓ Done")

results_df = pd.DataFrame(results).T.sort_values('Accuracy', ascending=False)
print("\n=== 5-FOLD CV RESULTS (Section 1 — Accuracy Only, No Class Balancing) ===")
print(results_df.to_string())


GPU available: False
Training Logistic Regression...
  ✓ Done
Training Random Forest...
  ✓ Done
Training XGBoost...
  ✓ Done
Training LightGBM...
  ✓ Done
Training KNN...
  ✓ Done

=== 5-FOLD CV RESULTS (Section 1 — Accuracy Only, No Class Balancing) ===
                     Accuracy  Balanced Acc  Sensitivity  Specificity     AUC
LightGBM               0.9005        0.6258       0.2713       0.9803  0.6258
Logistic Regression    0.8988        0.6061       0.2282       0.9840  0.6061
XGBoost                0.8967        0.6341       0.2953       0.9730  0.6341
Random Forest          0.8933        0.6309       0.2923       0.9696  0.6309
KNN                    0.8897        0.6226       0.2777       0.9674  0.6226


# **Hyperparameter Tuning for Top 3 Models (Maximizing Accuracy)**

Tune LightGBM, Random Forest, and XGBoost — the top three by CV accuracy — using `scoring='accuracy'`. No class-balancing techniques are applied.

In [32]:
tune_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── LightGBM (no SMOTE, no is_unbalance) ────────────────────────────────────
lgb_pipeline = Pipeline([
    ('model', lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, device=lgb_device))
])

lgb_param_grid = {
    'model__n_estimators':     [100, 200, 300],
    'model__learning_rate':    [0.05, 0.1],
    'model__num_leaves':       [31, 63, 127],
    'model__max_depth':        [-1, 10, 20],
    'model__subsample':        [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0],
}

lgb_search = RandomizedSearchCV(
    lgb_pipeline, lgb_param_grid,
    n_iter=16, cv=tune_cv, scoring='accuracy',
    random_state=42, n_jobs=1, verbose=1
)
print("Tuning LightGBM...")
lgb_search.fit(X_train_ready, y_train)
print(f"Best LGB Params:   {lgb_search.best_params_}")
print(f"Best LGB CV Score: {lgb_search.best_score_:.4f}")

# ── Random Forest (no SMOTE, no class_weight) ────────────────────────────────
rf_pipeline = Pipeline([
    ('model', RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_param_grid = {
    'model__n_estimators':      [100, 200, 300],
    'model__max_depth':         [10, 20, None],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf':  [1, 2],
    'model__max_features':      ['sqrt', 'log2'],
}

rf_search = GridSearchCV(
    rf_pipeline, rf_param_grid,
    cv=tune_cv, scoring='accuracy',
    n_jobs=1, verbose=1
)
print("\nTuning Random Forest...")
rf_search.fit(X_train_ready, y_train)
print(f"Best RF Params:   {rf_search.best_params_}")
print(f"Best RF CV Score: {rf_search.best_score_:.4f}")

# ── XGBoost — original code (no SMOTE, no scale_pos_weight) ─────────────────
xgb_pipeline = Pipeline([
    ('model', xgb.XGBClassifier(
        eval_metric='logloss', tree_method='hist',
        device=xgb_device, random_state=42, n_jobs=-1,
    ))
])

xgb_param_grid = {
    'model__n_estimators':     [100, 200, 300],
    'model__max_depth':        [3, 4, 5],
    'model__learning_rate':    [0.05, 0.1],
    'model__subsample':        [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0],
}

xgb_search = RandomizedSearchCV(
    xgb_pipeline, xgb_param_grid,
    n_iter=50, cv=tune_cv, scoring='accuracy',
    random_state=42, n_jobs=1, verbose=1
)
print("\nTuning XGBoost...")
xgb_search.fit(X_train_ready, y_train)
print(f"Best XGB Params:   {xgb_search.best_params_}")
print(f"Best XGB CV Score: {xgb_search.best_score_:.4f}")

# Summary
tuned_results = pd.DataFrame({
    'Model': ['Tuned LightGBM', 'Tuned Random Forest', 'Tuned XGBoost'],
    'Best CV Accuracy': [
        round(lgb_search.best_score_, 4),
        round(rf_search.best_score_, 4),
        round(xgb_search.best_score_, 4),
    ]
}).sort_values('Best CV Accuracy', ascending=False)
print("\n=== TUNED MODEL CV RESULTS ===")
print(tuned_results.to_string(index=False))

# Fit on full training data now so cell 16 can run after a kernel restart.
best_lgb = lgb_search.best_estimator_
best_rf  = rf_search.best_estimator_
best_xgb = xgb_search.best_estimator_

best_lgb.fit(X_train_ready, y_train)
best_rf.fit(X_train_ready, y_train)
best_xgb.fit(X_train_ready, y_train)

lgb_pred = best_lgb.predict(X_test_ready)
rf_pred  = best_rf.predict(X_test_ready)
xgb_pred = best_xgb.predict(X_test_ready)
print("\nBest estimators fitted on full training data. Predictions stored.")


Tuning LightGBM...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best LGB Params:   {'model__subsample': 1.0, 'model__num_leaves': 63, 'model__n_estimators': 100, 'model__max_depth': -1, 'model__learning_rate': 0.05, 'model__colsample_bytree': 1.0}
Best LGB CV Score: 0.9003

Tuning Random Forest...
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best RF Params:   {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 300}
Best RF CV Score: 0.9003

Tuning XGBoost...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best XGB Params:   {'model__subsample': 1.0, 'model__n_estimators': 300, 'model__max_depth': 3, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.8}
Best XGB CV Score: 0.9011

=== TUNED MODEL CV RESULTS ===
              Model  Best CV Accuracy
      Tuned XGBoost            0.9011
     Tuned LightGBM            0.9003
Tuned Random Forest     

In [33]:
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def lgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 127),
        'max_depth':         trial.suggest_int('max_depth', 5, 20),
        'subsample':         trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
    }
    model = lgb.LGBMClassifier(
        **params, random_state=42, n_jobs=1, verbose=-1, device=lgb_device
    )
    scores = cross_val_score(
        model, X_train_ready, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='accuracy', n_jobs=1
    )
    return scores.mean()

lgb_study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
lgb_study.optimize(lgb_objective, n_trials=75)

print(f"Best LGB Optuna params: {lgb_study.best_params}")
print(f"Best LGB Optuna CV accuracy: {lgb_study.best_value:.4f}")

best_lgb_optuna = lgb.LGBMClassifier(
    **lgb_study.best_params,
    random_state=42, n_jobs=-1, verbose=-1, device=lgb_device
)
best_lgb_optuna.fit(X_train_ready, y_train)
lgb_optuna_pred = best_lgb_optuna.predict(X_test_ready)

lgb_optuna_results = pd.DataFrame([{
    'Model':        'Optuna LightGBM',
    'Accuracy':     accuracy_score(y_test, lgb_optuna_pred),
    'Balanced Acc': balanced_accuracy_score(y_test, lgb_optuna_pred),
    'Sensitivity':  recall_score(y_test, lgb_optuna_pred, pos_label=1),
    'Specificity':  recall_score(y_test, lgb_optuna_pred, pos_label=0),
    'AUC':          roc_auc_score(y_test, best_lgb_optuna.predict_proba(X_test_ready)[:, 1]),
}]).round(4)
print(lgb_optuna_results.to_string(index=False))


Best LGB Optuna params: {'n_estimators': 379, 'learning_rate': 0.014768572792561604, 'num_leaves': 34, 'max_depth': 14, 'subsample': 0.7580206724151953, 'colsample_bytree': 0.7016943167776306, 'reg_alpha': 0.0036737032391772296, 'reg_lambda': 0.11461419722230076, 'min_child_samples': 12}
Best LGB Optuna CV accuracy: 0.9014
          Model  Accuracy  Balanced Acc  Sensitivity  Specificity    AUC
Optuna LightGBM    0.9024        0.6186       0.2522        0.985 0.7835


In [34]:
# Optuna hyperparameter search for XGBoost.
# Uses xgb.XGBClassifier (not XGBRFClassifier), no SMOTE, scoring='accuracy'.
# The original RandomizedSearchCV XGBoost (above) is retained separately.

def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'subsample':        trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    model = xgb.XGBClassifier(
        **params,
        eval_metric='logloss', tree_method='hist',
        device=xgb_device, random_state=42, n_jobs=1
    )
    scores = cross_val_score(
        model, X_train_ready, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='accuracy', n_jobs=1
    )
    return scores.mean()

xgb_study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=75)

print(f"Best XGB Optuna params: {xgb_study.best_params}")
print(f"Best XGB Optuna CV accuracy: {xgb_study.best_value:.4f}")

best_xgb_optuna = xgb.XGBClassifier(
    **xgb_study.best_params,
    eval_metric='logloss', tree_method='hist',
    device=xgb_device, random_state=42, n_jobs=-1
)
best_xgb_optuna.fit(X_train_ready, y_train)
xgb_optuna_pred = best_xgb_optuna.predict(X_test_ready)

xgb_optuna_results = pd.DataFrame([{
    'Model':        'Optuna XGBoost',
    'Accuracy':     accuracy_score(y_test, xgb_optuna_pred),
    'Balanced Acc': balanced_accuracy_score(y_test, xgb_optuna_pred),
    'Sensitivity':  recall_score(y_test, xgb_optuna_pred, pos_label=1),
    'Specificity':  recall_score(y_test, xgb_optuna_pred, pos_label=0),
    'AUC':          roc_auc_score(y_test, best_xgb_optuna.predict_proba(X_test_ready)[:, 1]),
}]).round(4)
print(xgb_optuna_results.to_string(index=False))


Best XGB Optuna params: {'n_estimators': 363, 'learning_rate': 0.04518615751682333, 'max_depth': 3, 'subsample': 0.9911631941429422, 'colsample_bytree': 0.9188161988312924, 'reg_alpha': 0.05548341813847414, 'reg_lambda': 0.12990132253334277, 'min_child_weight': 3}
Best XGB Optuna CV accuracy: 0.9009
         Model  Accuracy  Balanced Acc  Sensitivity  Specificity    AUC
Optuna XGBoost    0.9019        0.6159       0.2468       0.9851 0.7806


# **Model Testing (Maximizing Accuracy) — Section 1 Final Results**

In [35]:
# Section 1 Final Test Set Evaluation
# best_lgb/rf/xgb + predictions come from the tuning cell.
# best_lgb_optuna + lgb_optuna_pred come from the LightGBM Optuna cell.
# best_xgb_optuna + xgb_optuna_pred come from the XGBoost Optuna cell.

def eval_row(name, model, pred):
    return {
        'Model':        name,
        'Accuracy':     accuracy_score(y_test, pred),
        'Balanced Acc': balanced_accuracy_score(y_test, pred),
        'Sensitivity':  recall_score(y_test, pred, pos_label=1),
        'Specificity':  recall_score(y_test, pred, pos_label=0),
        'AUC':          roc_auc_score(y_test, model.predict_proba(X_test_ready)[:, 1]),
    }

final_results = pd.DataFrame([
    eval_row('Tuned LightGBM',      best_lgb,       lgb_pred),
    eval_row('Tuned Random Forest',  best_rf,        rf_pred),
    eval_row('Tuned XGBoost',        best_xgb,       xgb_pred),
    eval_row('Optuna LightGBM',      best_lgb_optuna, lgb_optuna_pred),
    eval_row('Optuna XGBoost',       best_xgb_optuna, xgb_optuna_pred),
]).round(4).sort_values('Accuracy', ascending=False)

print("\n=== SECTION 1 FINAL TEST SET RESULTS (Maximize Accuracy, No Class Balancing) ===")
print(final_results.to_string(index=False))



=== SECTION 1 FINAL TEST SET RESULTS (Maximize Accuracy, No Class Balancing) ===
              Model  Accuracy  Balanced Acc  Sensitivity  Specificity    AUC
    Optuna LightGBM    0.9024        0.6186       0.2522       0.9850 0.7835
     Optuna XGBoost    0.9019        0.6159       0.2468       0.9851 0.7806
Tuned Random Forest    0.9014        0.6011       0.2134       0.9888 0.7780
      Tuned XGBoost    0.9014        0.6161       0.2478       0.9844 0.7812
     Tuned LightGBM    0.9006        0.6156       0.2478       0.9834 0.7794


# **Section 2: Balanced Approach — SMOTE + Balanced Accuracy**

We re-introduce class-balancing techniques (SMOTE oversampling inside every pipeline fold, `class_weight='balanced_subsample'` for Random Forest, `scale_pos_weight` for XGBoost, `is_unbalance=True` for LightGBM) and switch the optimization target to **balanced accuracy** — the arithmetic mean of sensitivity and specificity. The goal: improve sensitivity (catching true "yes" subscriptions) without catastrophically sacrificing specificity.

In [36]:
# Section 2 standalone setup — run this cell if skipping Section 1
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import subprocess
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score, recall_score, roc_auc_score, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

# GPU detection
def gpu_available():
    try:
        subprocess.check_output(["nvidia-smi"], stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

USE_GPU = gpu_available()
xgb_device = "cuda" if USE_GPU else "cpu"
lgb_device  = "gpu"  if USE_GPU else "cpu"
print("GPU available:", USE_GPU)

neg, pos = np.bincount(y_train)
scale_pos = neg / pos

# Load data
train = pd.read_csv("./training_data.csv")
test  = pd.read_csv("./testing_data.csv")

# Preprocessing (identical to Section 1 cell 9)
train_df = train.copy()
test_df  = test.copy()

train_df.drop(columns=["duration"], inplace=True)
test_df.drop(columns=["duration"],  inplace=True)

train_df["was_contacted_before"] = (train_df["pdays"] != 999).astype(int)
test_df["was_contacted_before"]  = (test_df["pdays"]  != 999).astype(int)

cap_value = train_df["campaign"].quantile(0.95)
train_df["campaign"] = train_df["campaign"].clip(upper=cap_value)
test_df["campaign"]  = test_df["campaign"].clip(upper=cap_value)

train_df["y"] = (train_df["y"] == "yes").astype(int)
test_df["y"]  = (test_df["y"]  == "yes").astype(int)

X_train = train_df.drop(columns=["y"])
y_train = train_df["y"]
X_test  = test_df.drop(columns=["y"])
y_test  = test_df["y"]

cat_cols = X_train.select_dtypes(include="object").columns.tolist()
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=False)
X_test  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=False)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

X_train_ready = X_train.copy()
X_test_ready  = X_test.copy()

print(f"Ready — X_train: {X_train_ready.shape}, X_test: {X_test_ready.shape}")
print(f"Class distribution: {y_train.value_counts().to_dict()}")


GPU available: False
Ready — X_train: (32951, 63), X_test: (8237, 63)
Class distribution: {0: 29239, 1: 3712}


In [37]:
# Section 2: 5-fold CV with SMOTE + class balancing + balanced_accuracy scoring

cv_bal = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring_bal = {
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'accuracy':          make_scorer(accuracy_score),
    'sensitivity':       make_scorer(recall_score, pos_label=1),
    'specificity':       make_scorer(recall_score, pos_label=0),
    'auc':               make_scorer(roc_auc_score),
}

models_bal = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')),
    ]),
    'Random Forest': Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', RandomForestClassifier(
            n_estimators=100, random_state=42, n_jobs=-1
        )),
    ]),
    'XGBoost': Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', xgb.XGBClassifier(
            eval_metric='logloss', random_state=42, n_jobs=-1, device=xgb_device, scale_pos_weight=scale_pos
        )),
    ]),
    'LightGBM': Pipeline([
        ('model', lgb.LGBMClassifier(
            random_state=42, n_jobs=-1, verbose=-1,
            device=lgb_device, class_weight='balanced'
        )),
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
    ]),
}

results_bal = {}
for name, model in models_bal.items():
    print(f"Training {name} (balanced)...")
    cv_res = cross_validate(model, X_train_ready, y_train, cv=cv_bal, scoring=scoring_bal, n_jobs=-1)
    results_bal[name] = {
        'Balanced Acc': cv_res['test_balanced_accuracy'].mean().round(4),
        'Accuracy':     cv_res['test_accuracy'].mean().round(4),
        'Sensitivity':  cv_res['test_sensitivity'].mean().round(4),
        'Specificity':  cv_res['test_specificity'].mean().round(4),
        'AUC':          cv_res['test_auc'].mean().round(4),
    }
    print("  ✓ Done")

results_bal_df = pd.DataFrame(results_bal).T.sort_values('Balanced Acc', ascending=False)
print("\n=== 5-FOLD CV RESULTS (Section 2 — SMOTE + Balanced Accuracy) ===")
print(results_bal_df.to_string())


Training Logistic Regression (balanced)...
  ✓ Done
Training Random Forest (balanced)...
  ✓ Done
Training XGBoost (balanced)...
  ✓ Done
Training LightGBM (balanced)...
  ✓ Done
Training KNN (balanced)...
  ✓ Done

=== 5-FOLD CV RESULTS (Section 2 — SMOTE + Balanced Accuracy) ===
                     Balanced Acc  Accuracy  Sensitivity  Specificity     AUC
LightGBM                   0.7566    0.8462       0.6409       0.8723  0.7566
Logistic Regression        0.7465    0.8283       0.6409       0.8521  0.7465
XGBoost                    0.7325    0.8190       0.6210       0.8441  0.7325
KNN                        0.6886    0.7588       0.5981       0.7792  0.6886
Random Forest              0.6595    0.8884       0.3640       0.9550  0.6595


In [38]:
# Section 2: Hyperparameter tuning with SMOTE + balanced_accuracy scoring

tune_cv_bal = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── LightGBM (is_unbalance) ─────────────────────────────────────────
lgb_pipeline_bal = Pipeline([
    ('model', lgb.LGBMClassifier(
        random_state=42, n_jobs=-1, verbose=-1,
        device=lgb_device, is_unbalance=True
    )),
])

lgb_param_grid_bal = {
    'model__n_estimators':     [100, 200, 300],
    'model__learning_rate':    [0.05, 0.1],
    'model__num_leaves':       [31, 63, 127],
    'model__max_depth':        [-1, 10, 20],
    'model__subsample':        [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0],
}

lgb_search_bal = RandomizedSearchCV(
    lgb_pipeline_bal, lgb_param_grid_bal,
    n_iter=16, cv=tune_cv_bal, scoring='balanced_accuracy',
    random_state=42, n_jobs=1, verbose=1
)
print("Tuning LightGBM (balanced)...")
lgb_search_bal.fit(X_train_ready, y_train)
print(f"Best LGB Params:       {lgb_search_bal.best_params_}")
print(f"Best LGB Balanced Acc: {lgb_search_bal.best_score_:.4f}")

# ── Random Forest (SMOTE + balanced_subsample) ───────────────────────────────
rf_pipeline_bal = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(
        random_state=42, n_jobs=-1
    )),
])

rf_param_grid_bal = {
    'model__n_estimators':      [100, 200, 300],
    'model__max_depth':         [10, 20, None],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf':  [1, 2],
    'model__max_features':      ['sqrt', 'log2'],
}

rf_search_bal = GridSearchCV(
    rf_pipeline_bal, rf_param_grid_bal,
    cv=tune_cv_bal, scoring='balanced_accuracy',
    n_jobs=1, verbose=1
)
print("\nTuning Random Forest (balanced)...")
rf_search_bal.fit(X_train_ready, y_train)
print(f"Best RF Params:        {rf_search_bal.best_params_}")
print(f"Best RF Balanced Acc:  {rf_search_bal.best_score_:.4f}")

# ── XGBoost (SMOTE + scale_pos_weight) ──────────────────────────────────────
xgb_pipeline_bal = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', xgb.XGBClassifier(
        eval_metric='logloss', tree_method='hist',
        device=xgb_device, random_state=42, n_jobs=-1, scale_pos_weight=scale_pos
    )),
])

xgb_param_grid_bal = {
    'model__n_estimators':     [100, 200, 300],
    'model__max_depth':        [3, 4, 5],
    'model__learning_rate':    [0.05, 0.1],
    'model__subsample':        [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0],
}

xgb_search_bal = RandomizedSearchCV(
    xgb_pipeline_bal, xgb_param_grid_bal,
    n_iter=50, cv=tune_cv_bal, scoring='balanced_accuracy',
    random_state=42, n_jobs=1, verbose=1
)
print("\nTuning XGBoost (balanced)...")
xgb_search_bal.fit(X_train_ready, y_train)
print(f"Best XGB Params:       {xgb_search_bal.best_params_}")
print(f"Best XGB Balanced Acc: {xgb_search_bal.best_score_:.4f}")

# Summary
tuned_bal_summary = pd.DataFrame({
    'Model': ['Tuned LightGBM (Bal)', 'Tuned Random Forest (Bal)', 'Tuned XGBoost (Bal)'],
    'Best CV Balanced Acc': [
        round(lgb_search_bal.best_score_, 4),
        round(rf_search_bal.best_score_, 4),
        round(xgb_search_bal.best_score_, 4),
    ]
}).sort_values('Best CV Balanced Acc', ascending=False)
print("\n=== TUNED MODEL BALANCED CV RESULTS ===")
print(tuned_bal_summary.to_string(index=False))

# Fit on full training data
best_lgb_bal = lgb_search_bal.best_estimator_
best_rf_bal  = rf_search_bal.best_estimator_
best_xgb_bal = xgb_search_bal.best_estimator_

best_lgb_bal.fit(X_train_ready, y_train)
best_rf_bal.fit(X_train_ready, y_train)
best_xgb_bal.fit(X_train_ready, y_train)

lgb_pred_bal = best_lgb_bal.predict(X_test_ready)
rf_pred_bal  = best_rf_bal.predict(X_test_ready)
xgb_pred_bal = best_xgb_bal.predict(X_test_ready)
print("\nBest balanced estimators fitted on full training data.")


Tuning LightGBM (balanced)...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best LGB Params:       {'model__subsample': 1.0, 'model__num_leaves': 31, 'model__n_estimators': 100, 'model__max_depth': 10, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}
Best LGB Balanced Acc: 0.7568

Tuning Random Forest (balanced)...
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best RF Params:        {'model__max_depth': 10, 'model__max_features': 'log2', 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 300}
Best RF Balanced Acc:  0.7428

Tuning XGBoost (balanced)...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best XGB Params:       {'model__subsample': 0.8, 'model__n_estimators': 300, 'model__max_depth': 5, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}
Best XGB Balanced Acc: 0.7443

=== TUNED MODEL BALANCED CV RESULTS ===
                    Model  Best CV Balanced Acc
     Tuned LightGBM (Bal

In [39]:
# Optuna for LightGBM — Section 2 (balanced_accuracy, SMOTE inside objective)

def lgb_objective_bal(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 127),
        'max_depth':         trial.suggest_int('max_depth', 5, 20),
        'subsample':         trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
    }
    pipe = Pipeline([
        ('model', lgb.LGBMClassifier(
            **params, 
            random_state=42, n_jobs=1, verbose=-1, device=lgb_device, class_weight='balanced'
        )),
    ])
    scores = cross_val_score(
        pipe, X_train_ready, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='balanced_accuracy', n_jobs=1
    )
    return scores.mean()

lgb_study_bal = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
lgb_study_bal.optimize(lgb_objective_bal, n_trials=75)

print(f"Best LGB Optuna (balanced) params: {lgb_study_bal.best_params}")
print(f"Best LGB Optuna balanced accuracy: {lgb_study_bal.best_value:.4f}")

best_lgb_optuna_bal = Pipeline([
    ('model', lgb.LGBMClassifier(
        **lgb_study_bal.best_params, 
        random_state=42, n_jobs=-1, verbose=-1, device=lgb_device, class_weight='balanced'
    )),
])
best_lgb_optuna_bal.fit(X_train_ready, y_train)
lgb_optuna_pred_bal = best_lgb_optuna_bal.predict(X_test_ready)

lgb_optuna_bal_results = pd.DataFrame([{
    'Model':        'Optuna LightGBM (Bal)',
    'Accuracy':     accuracy_score(y_test, lgb_optuna_pred_bal),
    'Balanced Acc': balanced_accuracy_score(y_test, lgb_optuna_pred_bal),
    'Sensitivity':  recall_score(y_test, lgb_optuna_pred_bal, pos_label=1),
    'Specificity':  recall_score(y_test, lgb_optuna_pred_bal, pos_label=0),
    'AUC':          roc_auc_score(y_test, best_lgb_optuna_bal.predict_proba(X_test_ready)[:, 1]),
}]).round(4)
print(lgb_optuna_bal_results.to_string(index=False))


Best LGB Optuna (balanced) params: {'n_estimators': 332, 'learning_rate': 0.028925180569786615, 'num_leaves': 34, 'max_depth': 16, 'subsample': 0.7489848329703016, 'colsample_bytree': 0.719271721148873, 'reg_alpha': 0.6687024169536345, 'reg_lambda': 0.07947667604946006, 'min_child_samples': 42}
Best LGB Optuna balanced accuracy: 0.7588
                Model  Accuracy  Balanced Acc  Sensitivity  Specificity    AUC
Optuna LightGBM (Bal)    0.8399        0.7343       0.5981       0.8706 0.7804


In [40]:
# Optuna for XGBoost — Section 2 (balanced_accuracy, SMOTE inside objective)

def xgb_objective_bal(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'subsample':        trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    pipe = Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', xgb.XGBClassifier(
            **params, 
            eval_metric='logloss', tree_method='hist',
            device=xgb_device, random_state=42, n_jobs=1
        )),
    ])
    scores = cross_val_score(
        pipe, X_train_ready, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='balanced_accuracy', n_jobs=1
    )
    return scores.mean()

xgb_study_bal = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
xgb_study_bal.optimize(xgb_objective_bal, n_trials=75)

print(f"Best XGB Optuna (balanced) params: {xgb_study_bal.best_params}")
print(f"Best XGB Optuna balanced accuracy: {xgb_study_bal.best_value:.4f}")

best_xgb_optuna_bal = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', xgb.XGBClassifier(
        **xgb_study_bal.best_params, 
        eval_metric='logloss', tree_method='hist',
        device=xgb_device, random_state=42, n_jobs=-1
    )),
])
best_xgb_optuna_bal.fit(X_train_ready, y_train)
xgb_optuna_pred_bal = best_xgb_optuna_bal.predict(X_test_ready)

xgb_optuna_bal_results = pd.DataFrame([{
    'Model':        'Optuna XGBoost (Bal)',
    'Accuracy':     accuracy_score(y_test, xgb_optuna_pred_bal),
    'Balanced Acc': balanced_accuracy_score(y_test, xgb_optuna_pred_bal),
    'Sensitivity':  recall_score(y_test, xgb_optuna_pred_bal, pos_label=1),
    'Specificity':  recall_score(y_test, xgb_optuna_pred_bal, pos_label=0),
    'AUC':          roc_auc_score(y_test, best_xgb_optuna_bal.predict_proba(X_test_ready)[:, 1]),
}]).round(4)
print(xgb_optuna_bal_results.to_string(index=False))


Best XGB Optuna (balanced) params: {'n_estimators': 134, 'learning_rate': 0.010861912558070766, 'max_depth': 4, 'subsample': 0.9063870860287645, 'colsample_bytree': 0.9683408413358995, 'reg_alpha': 0.0019859938238980454, 'reg_lambda': 0.002757474501312354, 'min_child_weight': 10}
Best XGB Optuna balanced accuracy: 0.7329
               Model  Accuracy  Balanced Acc  Sensitivity  Specificity    AUC
Optuna XGBoost (Bal)    0.8598         0.722       0.5442       0.8998 0.7692


In [41]:
# Section 2 Final Test Set Evaluation + cross-section comparison

def eval_row_bal(name, model, pred):
    return {
        'Model':        name,
        'Accuracy':     accuracy_score(y_test, pred),
        'Balanced Acc': balanced_accuracy_score(y_test, pred),
        'Sensitivity':  recall_score(y_test, pred, pos_label=1),
        'Specificity':  recall_score(y_test, pred, pos_label=0),
        'AUC':          roc_auc_score(y_test, model.predict_proba(X_test_ready)[:, 1]),
    }

final_results_bal = pd.DataFrame([
    eval_row_bal('Tuned LightGBM (Bal)',      best_lgb_bal,         lgb_pred_bal),
    eval_row_bal('Tuned Random Forest (Bal)',  best_rf_bal,          rf_pred_bal),
    eval_row_bal('Tuned XGBoost (Bal)',        best_xgb_bal,         xgb_pred_bal),
    eval_row_bal('Optuna LightGBM (Bal)',      best_lgb_optuna_bal,  lgb_optuna_pred_bal),
    eval_row_bal('Optuna XGBoost (Bal)',       best_xgb_optuna_bal,  xgb_optuna_pred_bal),
]).round(4).sort_values('Balanced Acc', ascending=False)

print("\n=== SECTION 2 FINAL TEST SET RESULTS (SMOTE + Balanced Accuracy) ===")
print(final_results_bal.to_string(index=False))

# ── Section 1 vs Section 2: best model comparison (skipped if Section 1 not run) ──
try:
    best_s1 = final_results.sort_values('Accuracy', ascending=False).iloc[0].to_dict()
    best_s2 = final_results_bal.sort_values('Balanced Acc', ascending=False).iloc[0].to_dict()

    comparison = pd.DataFrame([
        {**best_s1, 'Section': 'Section 1 (Accuracy-First, No Balancing)'},
        {**best_s2, 'Section': 'Section 2 (SMOTE + Balanced Accuracy)'},
    ]).set_index('Section')[['Model', 'Accuracy', 'Balanced Acc', 'Sensitivity', 'Specificity', 'AUC']].round(4)

    print("\n=== SECTION 1 vs SECTION 2: BEST MODEL COMPARISON ===")
    print(comparison.to_string())
except NameError:
    print("\n(Section 1 results not available — skipping cross-section comparison.)")



=== SECTION 2 FINAL TEST SET RESULTS (SMOTE + Balanced Accuracy) ===
                    Model  Accuracy  Balanced Acc  Sensitivity  Specificity    AUC
    Optuna LightGBM (Bal)    0.8399        0.7343       0.5981       0.8706 0.7804
     Tuned LightGBM (Bal)    0.8407        0.7324       0.5927       0.8722 0.7812
Tuned Random Forest (Bal)    0.8603        0.7275       0.5560       0.8989 0.7764
      Tuned XGBoost (Bal)    0.8017        0.7251       0.6261       0.8241 0.7726
     Optuna XGBoost (Bal)    0.8598        0.7220       0.5442       0.8998 0.7692

=== SECTION 1 vs SECTION 2: BEST MODEL COMPARISON ===
                                                          Model  Accuracy  Balanced Acc  Sensitivity  Specificity     AUC
Section                                                                                                                  
Section 1 (Accuracy-First, No Balancing)        Optuna LightGBM    0.9024        0.6186       0.2522       0.9850  0.7835
Section 2 (

# **Impact of Including All Available Features on Model Performance**

In this section, we evaluate model performance when all available features are included. This allows us to compare predictive performance under different feature configurations and assess the impact of feature selection decisions.

In [ ]:
# WITH DURATION MODELING SECTION


# Copy raw data to avoid modifying originals
train_df_dur = train.copy()
test_df_dur = test.copy()

# Keep 'duration'
print("Duration column included:", 'duration' in train_df_dur.columns)


train_df_dur.drop(columns=['emp.var.rate', 'nr.employed'], inplace=True)
test_df_dur.drop(columns=['emp.var.rate', 'nr.employed'], inplace=True)


train_df_dur['was_contacted_before'] = (train_df_dur['pdays'] != 999).astype(int)
test_df_dur['was_contacted_before']  = (test_df_dur['pdays'] != 999).astype(int)


cap_value_dur = train_df_dur['campaign'].quantile(0.95)
train_df_dur['campaign'] = train_df_dur['campaign'].clip(upper=cap_value_dur)
test_df_dur['campaign']  = test_df_dur['campaign'].clip(upper=cap_value_dur)
print(f"Campaign capped at: {cap_value_dur}")

train_df_dur['y'] = (train_df_dur['y'] == 'yes').astype(int)
test_df_dur['y']  = (test_df_dur['y'] == 'yes').astype(int)


X_train_dur = train_df_dur.drop(columns=['y'])
y_train_dur = train_df_dur['y']
X_test_dur  = test_df_dur.drop(columns=['y'])
y_test_dur  = test_df_dur['y']


cat_cols_dur = X_train_dur.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns to encode: {cat_cols_dur}")

X_train_dur = pd.get_dummies(X_train_dur, columns=cat_cols_dur, drop_first=False)
X_test_dur  = pd.get_dummies(X_test_dur,  columns=cat_cols_dur, drop_first=False)


X_train_dur, X_test_dur = X_train_dur.align(X_test_dur, join='left', axis=1, fill_value=0)

X_train_dur_ready = X_train_dur.copy()
X_test_dur_ready  = X_test_dur.copy()

print(f"\nClass distribution before SMOTE: {y_train_dur.value_counts().to_dict()}")
print(f"Final shapes with duration — X_train: {X_train_dur_ready.shape}, X_test: {X_test_dur_ready.shape}")

Duration column included: True
Campaign capped at: 7.0
Categorical columns to encode: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']

Class distribution before SMOTE: {0: 29239, 1: 3712}
Final shapes with duration — X_train: (32951, 62), X_test: (8237, 62)


In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb

# Hyperparameter tuning for top 3 models with duration
# We use the same three models as before:
# 1. LightGBM
# 2. Random Forest
# 3. XGBoost

tune_cv_dur = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# Tune LightGBM
lgb_pipeline_dur = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', lgb.LGBMClassifier(
        random_state=42,
        n_jobs=-1,
        verbose=-1,
        device=lgb_device
    ))
])

lgb_param_grid_dur = {
    'model__n_estimators':     [100, 200, 300],
    'model__learning_rate':    [0.05, 0.1],
    'model__num_leaves':       [31, 63, 127],
    'model__max_depth':        [-1, 10, 20],
    'model__subsample':        [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}

lgb_search_dur = RandomizedSearchCV(
    estimator=lgb_pipeline_dur,
    param_distributions=lgb_param_grid_dur,
    n_iter=30,
    cv=tune_cv_dur,
    scoring='accuracy',
    random_state=42,
    n_jobs=1,
    verbose=1
)

print("\nTuning LightGBM with duration...")
lgb_search_dur.fit(X_train_dur_ready, y_train_dur)

print(f"Best LGB Params:   {lgb_search_dur.best_params_}")
print(f"Best LGB CV Score: {lgb_search_dur.best_score_:.4f}")


# Tune Random Forest
rf_pipeline_dur = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_param_grid_dur = {
    'model__n_estimators':      [100, 200, 300],
    'model__max_depth':         [10, 20, None],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf':  [1, 2],
    'model__max_features':      ['sqrt', 'log2']
}

rf_search_dur = RandomizedSearchCV(
    estimator=rf_pipeline_dur,
    param_distributions=rf_param_grid_dur,
    n_iter=30,
    cv=tune_cv_dur,
    scoring='accuracy',
    random_state=42,
    n_jobs=1,
    verbose=1
)

print("\nTuning Random Forest with duration...")
rf_search_dur.fit(X_train_dur_ready, y_train_dur)

print(f"Best RF Params:   {rf_search_dur.best_params_}")
print(f"Best RF CV Score: {rf_search_dur.best_score_:.4f}")


# Tune XGBoost
xgb_pipeline_dur = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', xgb.XGBClassifier(
        eval_metric='logloss',
        tree_method='hist',
        device=xgb_device,
        random_state=42,
        n_jobs=-1
    ))
])

xgb_param_grid_dur = {
    'model__n_estimators':     [100, 200, 300],
    'model__max_depth':        [3, 4, 5],
    'model__learning_rate':    [0.05, 0.1],
    'model__subsample':        [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}

xgb_search_dur = RandomizedSearchCV(
    estimator=xgb_pipeline_dur,
    param_distributions=xgb_param_grid_dur,
    n_iter=30,
    cv=tune_cv_dur,
    scoring='accuracy',
    random_state=42,
    n_jobs=1,
    verbose=1
)

print("\nTuning XGBoost with duration...")
xgb_search_dur.fit(X_train_dur_ready, y_train_dur)

print(f"Best XGB Params:   {xgb_search_dur.best_params_}")
print(f"Best XGB CV Score: {xgb_search_dur.best_score_:.4f}")


Tuning LightGBM with duration...
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best LGB Params:   {'model__subsample': 1.0, 'model__num_leaves': 31, 'model__n_estimators': 100, 'model__max_depth': 10, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}
Best LGB CV Score: 0.9164

Tuning Random Forest with duration...
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best RF Params:   {'model__n_estimators': 300, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': None}
Best RF CV Score: 0.9134

Tuning XGBoost with duration...
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best XGB Params:   {'model__subsample': 0.8, 'model__n_estimators': 300, 'model__max_depth': 4, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}
Best XGB CV Score: 0.9170


In [ ]:
# Final evaluation on untouched test set with duration
from sklearn.metrics import accuracy_score, recall_score, balanced_accuracy_score

# Extract best tuned models
best_lgb_dur = lgb_search_dur.best_estimator_
best_rf_dur  = rf_search_dur.best_estimator_
best_xgb_dur = xgb_search_dur.best_estimator_

# Fit best tuned models on full training data
best_lgb_dur.fit(X_train_dur_ready, y_train_dur)
best_rf_dur.fit(X_train_dur_ready, y_train_dur)
best_xgb_dur.fit(X_train_dur_ready, y_train_dur)

# Predict on untouched test set
lgb_pred_dur = best_lgb_dur.predict(X_test_dur_ready)
rf_pred_dur  = best_rf_dur.predict(X_test_dur_ready)
xgb_pred_dur = best_xgb_dur.predict(X_test_dur_ready)

final_results_dur = pd.DataFrame({
    'Model': ['Tuned LightGBM + Duration', 'Tuned Random Forest + Duration', 'Tuned XGBoost + Duration'],
    'Accuracy': [
        accuracy_score(y_test_dur, lgb_pred_dur),
        accuracy_score(y_test_dur, rf_pred_dur),
        accuracy_score(y_test_dur, xgb_pred_dur),
    ],
    'Balanced Acc': [
        balanced_accuracy_score(y_test_dur, lgb_pred_dur),
        balanced_accuracy_score(y_test_dur, rf_pred_dur),
        balanced_accuracy_score(y_test_dur, xgb_pred_dur),
    ],
    'Sensitivity': [
        recall_score(y_test_dur, lgb_pred_dur, pos_label=1),
        recall_score(y_test_dur, rf_pred_dur, pos_label=1),
        recall_score(y_test_dur, xgb_pred_dur, pos_label=1),
    ],
    'Specificity': [
        recall_score(y_test_dur, lgb_pred_dur, pos_label=0),
        recall_score(y_test_dur, rf_pred_dur, pos_label=0),
        recall_score(y_test_dur, xgb_pred_dur, pos_label=0),
    ],
    'AUC': [
        roc_auc_score(y_test_dur, best_lgb_dur.predict_proba(X_test_dur_ready)[:, 1]),
        roc_auc_score(y_test_dur, best_rf_dur.predict_proba(X_test_dur_ready)[:, 1]),
        roc_auc_score(y_test_dur, best_xgb_dur.predict_proba(X_test_dur_ready)[:, 1]),
    ],
}).round(4)

final_results_dur = final_results_dur.sort_values('Accuracy', ascending=False)

print("\n=== FINAL TEST SET RESULTS WITH DURATION ===")
print(final_results_dur.to_string(index=False))


AttributeError: 'RandomizedSearchCV' object has no attribute 'best_estimator_'